<a href="https://colab.research.google.com/github/TeflonOfJoy/projects/blob/main/H02A3A/exercises/E3_advanced_modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Imports
!pip install cpmpy numpy --quiet
!pip uninstall highspy -y # fix for temporary issue with ortools+highs
!pip install matplotlib

import cpmpy as cp
import numpy as np
import time
import matplotlib.pyplot as plt

def solve_with_time(model, time_limit=None):

    start_time = time.time()
    solution_found = model.solve(time_limit=time_limit)
    end_time = time.time()

    elapsed_time = end_time - start_time
    print("Time elapsed: {:.4f} seconds".format(elapsed_time))

    return solution_found

Found existing installation: highspy 1.15.1
Uninstalling highspy-1.15.1:
  Successfully uninstalled highspy-1.15.1


**Useful Resources:**
* CPMpy summary sheet: https://cpmpy.readthedocs.io/en/latest/summary.html
* CPMpy modeling documentation: https://cpmpy.readthedocs.io/en/latest/modeling.html

## **Session 4: Advanced Modelling**

### Outline:
1. Viewpoints
2. Channelling
3. Auxiliary Variables
4. Implied Constraints
5. Soft constraints
6. Multi-objective optimisation
7. Extra Exercises

## **1. Viewpoints**

In constraint programming, we often have multiple ways to represent the same problem. These representations, or "viewpoints," affect both how the CP solver explores the solution space and how easy it is to formulate the constraints. A carefully chosen viewpoint can:
- Reduce the complexity of the model, making it **faster** to solve
- Make the constraints **easier** to express
- **Implicitly** model constraints as part of the variables/domains definitions
- Improve **readability**

---

Consider the following problem:

We have $n$ pigeons and $m$ pigeonholes. Each pigeon must be assigned to one pigeonhole, but each pigeonhole can hold at most one pigeon.
Here, we consider $n=50$ pigeons and $m=60$ pigeonholes.

The output can be of the form:
```
Pigeon 1 goes to hole 1
Pigeon 2 goes to hole 7
...
Pigeon 50 goes to hole 53
```

Modelling this problem is quite straightforward. The focus is on constructing two models for this problem using **different viewpoints**.

### Viewpoint 1: Pigeons as variables

<details>
  <summary>Click to reveal hints</summary>

* We need a decision variable for each pigeon representing which hole it goes into.
* The [AllDifferent](https://cpmpy.readthedocs.io/en/latest/api/expressions/globalconstraints.html#cpmpy.expressions.globalconstraints.AllDifferent) constraint fits nicely when the pigeons are selected as decision variables.
</details>

In [ ]:
num_pigeons = 50
num_pigeonholes = 60

# Decision variables
pigeons = cp.intvar(1, num_pigeonholes, name="pigeons", shape=num_pigeons)
# Constraints
model_vp1 = cp.Model()

# Constraints
model_vp1.add(cp.AllDifferent(pigeons))

# Solve the model
solved = solve_with_time(model_vp1)

if solved:
  print(model_vp1.status())
  # Print the solution
  for index, pigeon in enumerate(pigeons):
    print(f"Pigeon {index+1} goes to hole {pigeon.value()}")
else:
  print("No solution found.")

Time elapsed: 0.0962 seconds
ExitStatus.FEASIBLE (0.09613299369812012 seconds)
Pigeon 1 goes to hole 50
Pigeon 2 goes to hole 49
Pigeon 3 goes to hole 48
Pigeon 4 goes to hole 47
Pigeon 5 goes to hole 46
Pigeon 6 goes to hole 45
Pigeon 7 goes to hole 44
Pigeon 8 goes to hole 43
Pigeon 9 goes to hole 42
Pigeon 10 goes to hole 41
Pigeon 11 goes to hole 40
Pigeon 12 goes to hole 39
Pigeon 13 goes to hole 38
Pigeon 14 goes to hole 37
Pigeon 15 goes to hole 36
Pigeon 16 goes to hole 35
Pigeon 17 goes to hole 34
Pigeon 18 goes to hole 33
Pigeon 19 goes to hole 32
Pigeon 20 goes to hole 31
Pigeon 21 goes to hole 30
Pigeon 22 goes to hole 29
Pigeon 23 goes to hole 28
Pigeon 24 goes to hole 27
Pigeon 25 goes to hole 26
Pigeon 26 goes to hole 25
Pigeon 27 goes to hole 24
Pigeon 28 goes to hole 23
Pigeon 29 goes to hole 22
Pigeon 30 goes to hole 21
Pigeon 31 goes to hole 20
Pigeon 32 goes to hole 19
Pigeon 33 goes to hole 18
Pigeon 34 goes to hole 17
Pigeon 35 goes to hole 16
Pigeon 36 goes to ho

### Viewpoint 2: Pigeonholes as variables

<details>
  <summary>Click to reveal hints</summary>

- Make sure that the domain includes also the option that the hole may be empty
- The constraint that each pigeon must be assigned to exactly one pigeonhole can be expressed in various ways, some of them are:
  - For each pigeon use `cp.sum` and ensure that it is equal to 1.
  - Use [GlobalCardinalityCount](https://cpmpy.readthedocs.io/en/latest/api/expressions/globalconstraints.html#cpmpy.expressions.globalconstraints.GlobalCardinalityCount)
  - Use [AllDifferentExcept0](https://cpmpy.readthedocs.io/en/latest/api/expressions/globalconstraints.html#cpmpy.expressions.globalconstraints.AllDifferentExcept0) and [Count](https://cpmpy.readthedocs.io/en/latest/api/expressions/globalfunctions.html#cpmpy.expressions.globalfunctions.Count)
</details>

In [ ]:
# Decision variables
pigeonholes = cp.intvar(0, num_pigeons, name="pigeonholes", shape=num_pigeonholes)

# Constraints
model_vp2 = cp.Model()

# TODO: Model the problem and print the solution.
model_vp2.add(cp.AllDifferentExcept0(pigeonholes))
model_vp2.add(cp.Count(pigeonholes, 0) == num_pigeonholes - num_pigeons)

# Solve the model
solved = solve_with_time(model_vp2)

if solved:
    print(model_vp2.status())
    # Print the solution
    for index, pigeonhole in enumerate(pigeonholes):
        print(f"Pigeon hole {index+1} has pigeon {pigeonhole.value()}")
else:
    print("No solution found.")


Time elapsed: 3.2053 seconds
ExitStatus.FEASIBLE (3.204995632171631 seconds)
Pigeon hole 1 has pigeon 1
Pigeon hole 2 has pigeon 2
Pigeon hole 3 has pigeon 3
Pigeon hole 4 has pigeon 4
Pigeon hole 5 has pigeon 5
Pigeon hole 6 has pigeon 6
Pigeon hole 7 has pigeon 7
Pigeon hole 8 has pigeon 8
Pigeon hole 9 has pigeon 9
Pigeon hole 10 has pigeon 10
Pigeon hole 11 has pigeon 11
Pigeon hole 12 has pigeon 12
Pigeon hole 13 has pigeon 13
Pigeon hole 14 has pigeon 14
Pigeon hole 15 has pigeon 15
Pigeon hole 16 has pigeon 16
Pigeon hole 17 has pigeon 17
Pigeon hole 18 has pigeon 18
Pigeon hole 19 has pigeon 19
Pigeon hole 20 has pigeon 20
Pigeon hole 21 has pigeon 21
Pigeon hole 22 has pigeon 22
Pigeon hole 23 has pigeon 23
Pigeon hole 24 has pigeon 24
Pigeon hole 25 has pigeon 25
Pigeon hole 26 has pigeon 26
Pigeon hole 27 has pigeon 27
Pigeon hole 28 has pigeon 28
Pigeon hole 29 has pigeon 29
Pigeon hole 30 has pigeon 30
Pigeon hole 31 has pigeon 31
Pigeon hole 32 has pigeon 32
Pigeon hole 3

**Observations:**
1. Which viewpoint was easier to model?

   The first model
2. Which viewpoint was faster to solve?

   The first model
3. Which constraints are implicitly covered in each viewpoint?

   Each pigeon is always implicitly assigned a hole

In [ ]:
# TODO: Count the time taken for model_vp1.solve() and model_vp2.solve()


## **2. Channeling**

Some constraints are more naturally expressed in one viewpoint, while other constraints of the same model are maybe more naturally expressed in another viewpoint. In such cases, **channeling** is a useful technique that connects variables from different viewpoints within a single model. Some key benefits:

1. You can use the most intuitive viewpoint for each constraint.
2. The solver may deduce information across linked viewpoints, potentially speeding up the solving process.
3. The final models can become easier to read, maintain, and extend.

Channeling is achieved by adding constraints that ensure consistency between the decision variables of different viewpoints.

---

Consider the *Student Seating Problem* from the lecture:

You are tasked with finding an optimal seating arrangement for `nStudents = 15` students across `nTables = 5` tables, where there are `nChairs = 20` chairs in total. You are given a list of chair assignments for each table:
`tables = [0..3, 4..7, 8..11, 12..15, 16..19]` (as in the lecture).
The goal is to assign **all** students to the chairs in such a way that each table has either at least half or no chairs occupied.

Create variables with two different viewpoints and try to model each constraint with the viewpoint that is easier. Before solving, it is necessary that the variables between the viewpoints are logically connected.

<details>
  <summary>Click to reveal hints</summary>

There are two high-level constraints that need to be satisfied:
1. From a student perspective: **All** students sit on a **different** chair
2. From a chair perspective: In each table, either **all chairs are empty** or **at least half are non-empty**.
</details>

In [ ]:
# Data
n_students = 15
n_chairs = 20
n_tables = 5

# Chair indexes for each table
# [0..3, 4..7, 8..11, 12..15, 16..19]
# np.array_split splits range(n_chairs) into n_tables consecutive chunks of chairs
tables = list(np.array_split(range(n_chairs), n_tables))

# Model
model = cp.Model()

# Viewpoint 1: Students as vars
students = cp.intvar(0, n_chairs-1, name="students", shape=n_students)  # TODO

# TODO: Add the constraint(s) that is (are) easier to express with vp1
model.add(cp.AllDifferent(students))
# Viewpoint 2: Chairs as vars
chairs = cp.intvar(-1, n_students, name="chairs", shape=n_chairs)  # 0 as empty

# TODO: Add the constraint(s) that is (are) easier to express with vp2
cp.AllDifferentExceptN(chairs, -1)
for table in tables:
    occ = cp.sum(chairs[table] != -1)
    model.add((occ == 0) | (occ >= len(table) // 2))
# Channeling: At this point we have modeled half of the constraints with the
# vars of vp1 and half with those of vp2. We need to merge the two viewpoints.

# TODO: Connect the students and chairs viewpoints
for i in  range(n_students):
    chairs[students[i]] == i

# TODO: Solve the model and print the solution


## **3. Auxiliary Variables**

Auxiliary variables can help in expressing complex constraints more naturally and clearly. They can be useful when establishing relationships between decision variables and additional properties or constraints, making it easier to formulate constraints. So, they are used:

* Because it is difficult to express the constraints in terms of the existing variables, or
* To allow the constraints to be expressed in a form that would propagate better.

**Car Sequencing**

Imagine an assembly line producing different types of cars, each with optional features (e.g. air-conditioning, sunroof etc.). Each station on the assembly line handles a specific feature and has a limited capacity, meaning it can only work on a certain percentage of the cars that pass through.

To prevent any station from being overwhelmed, cars must be arranged in a sequence so that the number of cars needing a particular feature never exceeds the station's capacity. For example, if a station can only handle half of the cars that pass by, then at most 1 out of every 2 cars in the sequence should require that feature.

A model of this problem is also available in the slides of lecture 4, slide 19.
Try to model it without looking at the slides first.

In [ ]:
# Data
at_most = [1, 2, 2, 2, 1]  # The maximum number of times an option can be present in a group of consecutive timeslots (see 'per_slots' below)
per_slots = [2, 3, 3, 5, 5]  # The number of consecutive timeslots (window) in which an option has a limit (see 'at_most' above)

demand = [1, 1, 2, 2, 2, 2]  # The demand per type of car
requires = cp.cpm_array([
    [1, 0, 1, 1, 0],
    [0, 0, 0, 1, 0],
    [0, 1, 0, 0, 1],
    [0, 1, 0, 1, 0],
    [1, 0, 1, 0, 0],
    [1, 1, 0, 0, 0]])  # Which options (columns) are needed for each car type (rows).

n_cars = sum(demand)  # The amount of cars to sequence
n_options = len(at_most)  # The amount of different options
n_types = len(demand)  # The amount of different car types

# Decision Variables
sequence = cp.intvar(0, n_types-1, name="sequence", shape=n_cars)  # The sequence of cars

# TODO: Define the auxiliary variables -> Which options are needed for each car in the sequence.
option = cp.boolvar(name="option", shape=(n_cars, n_options))

# Model
car_seq_model = cp.Model()

# TODO: The amount of each type of car in the sequence has to be equal to the demand for that type
for i in range(n_types):
    car_seq_model.add(cp.sum(sequence == i) == demand[i])

# TODO: Make sure that the options in the option table correspond to those of the car type
for i in range(n_cars):
  for j in range(n_options):
    car_seq_model.add(option[i, j] == requires[sequence[i], j])

# TODO: Check that no more than "at most" car options are used per "per_slots" slots
for j in range(n_options):
  for i in range(n_cars - per_slots[j] + 1):
    car_seq_model.add(cp.sum(option[i:i+per_slots[j], j]) <= at_most[j])

# TODO: Solve and print the sequence of cars and which options are required per car.
if car_seq_model.solve():
    print("Sequence: ", " ".join(map(str, sequence.value())))
    print("-" * 30)
    for j in range(n_options):
        print(f"Option {j+1}: ", " ".join(map(str, option[:, j].value().astype(int))))
else:
    print("No solution found!")

Sequence:  0 2 5 1 5 3 4 2 3 4
------------------------------
Option 1:  1 0 1 0 1 0 1 0 0 1
Option 2:  0 1 1 0 1 1 0 1 1 0
Option 3:  1 0 0 0 0 0 1 0 0 1
Option 4:  1 0 0 1 0 1 0 0 1 0
Option 5:  0 1 0 0 0 0 0 1 0 0


For reference, here is a solution of this problem:

```
Sequence:  0 2 5 1 5 3 4 2 3 4
------------------------------
Option 1:  1 0 1 0 1 0 1 0 0 1
Option 2:  0 1 1 0 1 1 0 1 1 0
Option 3:  1 0 0 0 0 0 1 0 0 1
Option 4:  1 0 0 1 0 1 0 0 1 0
Option 5:  0 1 0 0 0 0 0 1 0 0
```

## **4. Implied Constraints**

As taken from the lecture slides:

Can we do something to make our model quicker?
During the search process, the solver will explore several infeasible parts of the search tree. Can we avoid that?
Although implied constraints are logically redundant, and do not change the set of solutions, they can be used to reduce the search effort of the solving process.

Consider the **Magic Series problem**:

The element at index i in I = 0..(n-1) is the number of occurrences of i, for an integer n. Create a model that finds a magic series for n = 100.

As seen in lecture 4 slide 24.

Print the time taken to solve this model.

In [5]:
n = 100

# TODO: Decision Variables
Magic = cp.intvar(0, n-1, name="magic", shape=n)

# Model
model = cp.Model()

# TODO: Add the constraint
for i in range(n):
  model.add(Magic[i] == cp.sum(Magic == i))

# TODO: Solve the model and print the solution (print the time taken)

model.solve()
print(Magic.value())
print(model.status())

[96  2  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  1  0  0  0]
ExitStatus.FEASIBLE (12.780357599258423 seconds)


We can significantly improve solving time by adding implied constraints.

Add implied constraints (try it initially without looking at them in the slides), and compare the time taken to solve the new model with the above one.

In [4]:
# TODO: Implied constraints
model.add(cp.sum(Magic) == n)


# TODO: Solve the model and print the solution (print the time taken)

model.solve()
print(Magic.value())
print(model.status())

[96  2  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  1  0  0  0]
ExitStatus.FEASIBLE (6.588680267333984 seconds)


## **5. Soft Constraints**

A local bakery plans its production for the week (Tuesday to Sunday). It bakes 4 products: bread, baguettes, croissants and pies. Products are baked in **batches**. Each batch consists of only one type of product (two different products are never baked together) and each oven batch always yields a fixed number of items depending on the product type (e.g. one batch fits 20 loaves of bread or 30 baguettes).

The bakery has the following requirements:
1. **Oven:** the oven can bake at most `oven_slots` batches per day.
2. **Demand:** the bakery wants to match the customer demand for each product on each day.

The data is given below.

In [6]:
# Data
days = ["Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
products = ["bread", "baguette", "croissant", "pie"]

batch_size = np.array([20, 30, 24, 6])  # items produced by one oven batch; 1 batch of croissant = 24 croissants
oven_slots = 10                         # oven batches per day

# Demand in items, rows = products, columns = days
demand = np.array([
    # Tue  Wed  Thu  Fri  Sat  Sun
    [ 48,  46,  48,  48,  68,  88],   # bread
    [ 27,  26,  27,  27,  38,  65],   # baguette
    [ 22,  21,  22,  22,  30,  52],   # croissant
    [  5,   5,   5,   5,   8,  13],   # pie
])

n_products, n_days = demand.shape

In [7]:
# Helper to print a production plan
# Shows production minus demand per product and day (negative = shortage, positive = leftovers)
def print_plan(batches):
    b = np.asarray(batches)
    diff = b * batch_size[:, None] - demand

    print(f"{'':10}" + "".join(f"{d:>6}" for d in days))
    for p in range(n_products):
        print(f"{products[p]:10}" + "".join(f"{x:>+6}" for x in diff[p]))
    print()
    print(f"Total shortage:  {-diff[diff < 0].sum()} items")
    print(f"Total leftovers: {diff[diff > 0].sum()} items")
    print(f"Idle oven slots: {oven_slots * n_days - b.sum()} / {oven_slots * n_days}")

### Step 1: The strict model

The baker hates waste, so the house rule is: **never bake more than what is demanded** (no leftovers).
Under this rule, the bakery wants to sell as much as possible.

In [12]:
# Decision variables
batches = cp.intvar(0, oven_slots, shape=(n_products, n_days), name="batches")

# Model
model = cp.Model()

# TODO: Constraints
# 1. Oven capacity
model.add(cp.sum(batches) <= oven_slots * n_days)
# 2. House rule: no leftovers
for i in range(n_products):
  for j in range(n_days):
    model.add(batches[i, j] <= demand[i, j])

# TODO: Objective: sell as much as possible
model.maximize(cp.sum(batches))

# Solve
if model.solve():
    print_plan(batches.value())

             Tue   Wed   Thu   Fri   Sat   Sun
bread        -48   -46   -48   -48   -68   -88
baguette     -27   -26   -27   -27   -38   -65
croissant    -22   -21   -22   +26  +210  +188
pie          +25   +25   +25   +25   +40   +47

Total shortage:  621 items
Total leftovers: 611 items
Idle oven slots: 0 / 60


### Step 2: Soft demand with a single slack variable

The "no leftovers" rule is clearly too strict: because of the batch sizes, most demands cannot be met without some leftovers. What the bakery really wants is to produce **as close as possible** to the demand; not too much, not too little.

<details>
  <summary>Click to reveal hints</summary>

* Introduce one slack variable per product and day, with a domain that includes negative values.
* `production == demand + slack`
* Minimize the amount of slack.
</details>

In [13]:
# TODO: Decision variables, including the slack variables
batches = cp.intvar(0, oven_slots, shape=(n_products, n_days), name="batches")
slack = cp.intvar(-oven_slots, 0, shape=(n_products, n_days), name="slack")

# Model
model = cp.Model()

# TODO: Constraints
# 1. Oven capacity (hard)
model.add(cp.sum(batches) <= oven_slots * n_days)
# 2. Demand (soft)
model.add(batches == demand + slack)

# TODO: Objective: minimize the slack
model.minimize(cp.sum(slack))

if model.solve():
    print_plan(batches.value())

### Step 3: Separate slack for shortage and leftovers

A shortage is worse than a leftover, but with a single slack variable both directions are penalized equally.

<details>
  <summary>Click to reveal hints</summary>

* `production == demand + over - under`
* Minimize `P_over * cp.sum(over) + P_under * cp.sum(under)`.
* Try `P_over = 1` and different values for `P_under` (e.g. 2, 3, 5).
</details>

In [ ]:
# TODO: Decision variables, with separate slack variables for leftovers and shortage
batches = cp.intvar(0, oven_slots, shape=(n_products, n_days), name="batches")
over = None
under = None

# Model
model = cp.Model()

# TODO: Constraints
# 1. Oven capacity (hard)

# 2. Demand (soft), with separate slack per direction


# TODO: Objective: penalize shortage more than leftovers
P_over, P_under = 1, 1


if model.solve():
    print_plan(batches.value())

### Step 4: Soft global constraints

Before the shop opens on Saturday, the bakers have to prepare all products. Each preparation task takes a number of hours and needs a number of bakers at the same time.
There are 2 bakers. They start at 4:00 and the shop opens at 8:00, so all tasks must be finished within 4 hours.

This is a scheduling problem that can be modelled with the [Cumulative](https://cpmpy.readthedocs.io/en/latest/api/expressions/globalconstraints.html#cpmpy.expressions.globalconstraints.Cumulative) global constraint.

In [ ]:
# Data
duration = [2, 2, 3, 2, 1]        # hours per task (bread, baguettes, croissants, pies, cakes)
bakers_needed = [1, 1, 2, 1, 2]   # bakers needed per task
bakers = 2
deadline = 4

# Decision variables
start = cp.intvar(0, deadline, shape=len(duration), name="start")
end = cp.intvar(0, deadline, shape=len(duration), name="end")

# TODO: 1. Hard version, using the Cumulative global constraint
hard_model = cp.Model()

print(hard_model.solve())

The schedule will not be solvable. Solve the problem by introducing some slack on the Cumulative global.

In [ ]:
# TODO: 2. Soft version: add slack to the Cumulative constraint (without decomposing it)
extra = None # extra bakers (slack variables)
soft_model = cp.Model()

# TODO: Objective


if soft_model.solve():
    print("Extra helpers:", extra.value())
    print("Start times:  ", start.value())

## **6. Multi-objective optimisation**

It is Friday evening at the pizzeria. There are 12 orders waiting to be delivered, and each order takes a rider a number of minutes (drive there and back).
Up to 5 riders are available. Each order is delivered by one rider; a rider delivers their orders one after the other.

The pizzeria has two objectives:
1. Use as **few riders** as possible (each rider costs money).
2. Deliver all orders as **fast** as possible: minimize the time until the last order is delivered (the longest working time of a rider).

These objectives conflict: more riders means faster delivery, but higher cost.

In [ ]:
# Data
duration = [35, 30, 25, 25, 20, 20, 15, 15, 10, 10, 10, 5]  # minutes per order
max_riders = 5

# Base model
def pizza_model():
    # Decision variables
    rider = cp.intvar(0, max_riders - 1, shape=len(duration), name="rider")  # rider of each order

    # Calculate the time for each rider
    time = [cp.sum([d * (rider[o] == r) for o, d in enumerate(duration)]) for r in range(max_riders)]

    # Objectives
    # 1: Minimize the number of riders
    n_riders = cp.sum([t > 0 for t in time])
    # 2: Minimize the finish time
    finish_time = cp.max(time)

    return cp.Model(), n_riders, finish_time

# Helper function for visualizing the Pareto front
def plot_pareto(solutions):
    """Plot the Pareto front (fastest finish time per number of riders) and the given solutions."""
    front = []
    for k in range(1, max_riders + 1):
        model, n_riders, finish_time = pizza_model()
        model.add(n_riders == k)
        model.minimize(finish_time)
        model.solve()
        front.append((k, finish_time.value()))
    plt.plot(*zip(*front), "o-", color="lightgray", label="Pareto front")

    for name, (r, t) in solutions.items():
        print(f"{name:18} riders = {r}, finish time = {t} min")
        plt.plot(r, t, "*" if "lex" in name else "x", markersize=12, label=name)
    plt.xlabel("number of riders")
    plt.xticks(range(1, max_riders + 1))
    plt.ylabel("finish time (min)")
    plt.legend()
    plt.show()

The model and a helper to plot the Pareto front are given above. `pizza_model()` returns a model, together with the two objectives `n_riders` and `finish_time`.

**Lexicographic optimisation:** rank the objectives and optimize them one after the other. First minimize the number of riders, **fix** its optimal value with a constraint, then minimize the finish time. Also do it in the reverse order: first the finish time, then the number of riders.

Store each solution in the dictionary `solutions` as `solutions[name] = (n_riders, finish_time)`.

In [ ]:
def lexicographic_pizza(solutions):
    # TODO: First minimize the number of riders, fix it, then minimize the finish time
    model, n_riders, finish_time = pizza_model()


    solutions["lex: riders first"] = (n_riders.value(), finish_time.value())

    # TODO: The reverse order: first the finish time, then the number of riders
    model, n_riders, finish_time = pizza_model()


    solutions["lex: time first"] = (n_riders.value(), finish_time.value())

**Weighted sum:** combine both objectives into a single objective. Try a few different weights.

In [ ]:
def weighted_pizza(solutions, weights):
    model, n_riders, finish_time = pizza_model()
    w1, w2 = weights

    # TODO: Minimize the weighted sum of both objectives and solve


    solutions[f"weighted w={w1}, {w2}"] = (n_riders.value(), finish_time.value())

In [ ]:
# Visualize the Pareto front
solutions = {}

lexicographic_pizza(solutions)
for weights in [(1, 1), (15, 1), (30, 1), (60, 1)]:
    weighted_pizza(solutions, weights)

plot_pareto(solutions)

## **7. Extra Exercises**


### Auxiliary Variables: Orthogonal Latin Squares

Consider the [Orthogonal Latin Squares](https://en.wikipedia.org/wiki/Mutually_orthogonal_Latin_squares) problem:

A Latin square is an $ n \times n $ array filled with $n$ different symbols, each occurring exactly once in each row and exactly once in each column. In other words, a Latin Square is a square grid where:

* Each row contains all $n$ distinct symbols exactly once.
* Each column contains all $n$ distinct symbols exactly once.
* No symbol appears twice in the same row or column.

Two Latin squares are said to be orthogonal if, when placed on top of each other, each possible combination of their symbols appears exactly once.

**Instructions:**

You have a set of $ n $ integers (0..n-1), and your task is to construct two $ n \times n $ Latin squares that are orthogonal.

1. Define decision variables for the Latin squares.
2. Introduce auxiliary variables to represent each cell's pairing from both Latin squares.
3. Ensure that each pairing is unique across the entire grid.

In [ ]:
n = 7  # Size of the Latin square, and number of symbols

# TODO: Decision variables
square1 = None
square2 = None

# TODO: Auxiliary variables to combine symbols from both squares
pairing = None

# Model
model = cp.Model()

# TODO: Latin square constraints

# TODO: Connect the pairing variables with the square variables

# TODO: All pairings should be unique


# TODO: Solve the model and print the squares


Now, to see the benefit of auxiliary variables, try to model the same problem but without using `pairings`. The orthogonality constraint needs to be formulated directly over the square variables. This is more complicated to model, and possibly also slower for the solver (check the run time).

In [ ]:
n = 7  # Size of the Latin square, and number of symbols

# Decision variables for two Latin squares
square1 = None
square2 = None

# Model
model_no_aux = cp.Model()

# TODO: Latin square constraints

# TODO: Orthogonality constraint: Ensure each combination is unique by pair-wise comparing all pairs


# TODO: Solve the model and print the time taken. Print the time taken also for the previous model that contains auxiliary variables.
